In [5]:
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from llava.conversation import conv_templates, SeparatorStyle
from llava.model.builder import load_pretrained_model
from llava.utils import disable_torch_init
from llava.mm_utils import tokenizer_image_token, get_model_name_from_path, KeywordsStoppingCriteria, process_images

In [6]:
import argparse
import torch
import os
import json
from tqdm import tqdm
import shortuuid
from PIL import Image
import math
from transformers import set_seed, logging

In [4]:
from llava.model.builder import load_pretrained_model
tokenizer, model, image_processor, context_len = load_pretrained_model(
        model_path='/work/vlmwork/LLaVA-Med/llavamodel',
        model_base=None,
        model_name='llava-med-v1.5-mistral-7b'
 )


/root/miniconda/envs/llava-med/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 4/4 [00:13<00:00,  3.37s/it]
Some weights of the model checkpoint at /work/vlmwork/LLaVA-Med/llavamodel were not used when initializing LlavaMistralForCausalLM: ['model.vision_tower.vision_tower.vision_model.encoder.layers.5.self_attn.q_proj.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.13.self_attn.out_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.22.mlp.fc1.bias', 'model.vision_tower.vision_tower.vision_model.encoder.layers.9.self_attn.k_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.layers.5.self_attn.v_proj.weight', 'model.vision_tower.vision_tower.vision_model.encoder.l

In [31]:
qs = "Diagnose the image if it has a condition, otherwise say so."
qs = qs.replace(DEFAULT_IMAGE_TOKEN, '').strip()

if model.config.mm_use_im_start_end:
    qs = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + qs
else:
    qs = DEFAULT_IMAGE_TOKEN + '\n' + qs

conv = conv_templates['mistral_instruct'].copy()
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()    

input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda()

IMAGE_FOLDER = os.path.join(f'data/vlmdata')
image_file = 'img_0.png'
image = Image.open(os.path.join(IMAGE_FOLDER, image_file))

image_tensor = process_images([image], image_processor, model.config)[0]

stop_str = conv.sep if conv.sep_style != SeparatorStyle.TWO else conv.sep2
keywords = [stop_str]
stopping_criteria = KeywordsStoppingCriteria(keywords, tokenizer, input_ids)



In [32]:
prompt

'[INST] <image>\nDiagnose the image if it has a condition, otherwise say so. [/INST]'

In [33]:
with torch.inference_mode():
    output_ids = model.generate(
        input_ids,
        images=image_tensor.unsqueeze(0).half().cuda(),
        do_sample=True,
        temperature=0.2,
        top_p=None,
        num_beams=1,
        # no_repeat_ngram_size=3,
        max_new_tokens=1024,
        use_cache=True
    )

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [34]:
outputs = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()

In [35]:
outputs

'The image is a fundus photograph, which is a type of retinal examination. In this case, the fundus photograph is from a patient with a condition called retinal detachment. Retinal detachment is a serious eye condition where the retina, the light-sensitive layer of tissue at the back of the eye, separates from the underlying supportive tissue. This can lead to vision loss if not treated promptly.'